# KPI 7 — Geographic Sales

**requirements.md (Customer Intelligence, KPI 7):** "Geographic sales: revenue & order count by country/city; growth regions."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_7_geographic_sales` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cell below once (or after any change to the fact/dim definitions).

**Definitions (state formulas):**

* **Grain:** one row per **(country, city)** of buying companies — the 11 geographies of the 390 companies with ≥ 1 valid-date order line (KPI 2 convention; all `company_type = 'buyer'`). Geography (`country`, `city`, verbatim casing) comes from the current dim version (`__END_AT IS NULL`) joined via the fact's as-of `company_sk`, so each line is attributed to the company version valid when the order was placed. The country rollup is a `GROUP BY country` over this view (same pattern as KPI 3's category rollup).
* **Revenue bases (KPI 1 convention):** `gross_revenue` = `SUM(line_total)` over ALL statuses with `order_date_valid = true`; `net_revenue` = same excluding `status = 'cancelled'`. Totals reconcile exactly with KPI 1–6 (net 28,333,498.48 / gross 30,723,269.53).
* **Order count (KPI 5/6 convention):** `order_count` = distinct valid-date orders of any status; `SUM(order_count)` over the view = 10,817 (every order belongs to exactly one company, hence one geography).
* **Growth regions:** the month containing the latest valid order date (2026-11, partial through Nov 6) is excluded from BOTH windows; `net_ttm` / `order_count_ttm` = trailing 12 **full** months (currently 2025-11..2026-10), `*_prior_ttm` = the 12 months before that (2024-11..2025-10). Window boundaries are computed dynamically from `MAX(order_date)`, never hardcoded. `*_yoy_growth_pct` = `100 * (ttm − prior) / prior`; NULL when prior = 0 (a region too new to have a comparable base).
* **Share:** `net_revenue_share_pct` = `100 * net_revenue / total net revenue` (sums to 100.00 over the view).

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_7_geographic_sales
TBLPROPERTIES (
  'comment' = 'KPI 7 - Geographic sales (requirements.md: revenue & order count by country/city; growth regions). Grain: one row per (country, city) of BUYING companies - companies with >= 1 valid-date order line (KPI 2 convention); 11 geographies / 390 buying companies. Geography (verbatim casing) from the current SCD2 dim_companies version (__END_AT IS NULL) joined via the fact''s as-of company_sk, so each line is attributed to the company version valid when the order was placed; country rollup = GROUP BY country over this view. gross_revenue = SUM(line_total) ALL statuses, order_date_valid = true; net_revenue = same excluding status = ''cancelled'' (KPI 1 bases; totals reconcile exactly with KPI 1-6: net 28,333,498.48 / gross 30,723,269.53). order_count = COUNT(DISTINCT order_id) valid-date orders of any status (KPI 5/6 convention); SUM(order_count) = 10,817. net_revenue_share_pct = 100 * net / total net (sums to 100.00). Growth: the month containing the latest valid order date is excluded from BOTH windows (it may be partial); window boundaries computed dynamically from MAX(order_date): net_ttm / order_count_ttm = trailing 12 FULL months, *_prior_ttm = the 12 months before that; *_yoy_growth_pct = 100*(ttm - prior)/prior, NULL when prior = 0 (region too new for a comparable base).'
)
AS
WITH fact AS (
  SELECT company_sk, order_id, status, line_total, order_date
  FROM b2b_ecommerce.gold.fact_order_items
  WHERE order_date_valid = true
),
bounds AS (
  -- trunc_month = first day of the month containing the latest valid order. That
  -- month may be partial (currently 2026-11 through Nov 6), so both growth
  -- windows use months strictly before it: TTM = [trunc-12m, trunc) and
  -- prior TTM = [trunc-24m, trunc-12m). Re-derived on every query.
  SELECT DATE_TRUNC('month', MAX(order_date)) AS trunc_month
  FROM fact
),
lines AS (
  SELECT
    d.country,
    d.city,
    d.company_id,
    f.order_id,
    f.line_total,
    CASE WHEN f.status <> 'cancelled' THEN f.line_total ELSE 0 END AS net_amt,
    f.order_date >= ADD_MONTHS(b.trunc_month, -12)
      AND f.order_date <  b.trunc_month                  AS in_ttm,
    f.order_date >= ADD_MONTHS(b.trunc_month, -24)
      AND f.order_date <  ADD_MONTHS(b.trunc_month, -12) AS in_prior
  FROM fact f
  JOIN b2b_ecommerce.gold.dim_companies d
    ON d.company_sk = f.company_sk
   AND d.__END_AT IS NULL
  CROSS JOIN bounds b
)
SELECT
  country,
  city,
  COUNT(DISTINCT company_id) AS buying_companies,
  COUNT(DISTINCT order_id)   AS order_count,
  ROUND(SUM(line_total), 2) AS gross_revenue,
  ROUND(SUM(net_amt), 2)    AS net_revenue,
  ROUND(100 * SUM(net_amt) / SUM(SUM(net_amt)) OVER (), 2) AS net_revenue_share_pct,
  ROUND(SUM(CASE WHEN in_ttm   THEN net_amt ELSE 0 END), 2) AS net_ttm,
  ROUND(SUM(CASE WHEN in_prior THEN net_amt ELSE 0 END), 2) AS net_prior_ttm,
  ROUND(100 * (SUM(CASE WHEN in_ttm   THEN net_amt ELSE 0 END)
             - SUM(CASE WHEN in_prior THEN net_amt ELSE 0 END))
        / NULLIF(SUM(CASE WHEN in_prior THEN net_amt ELSE 0 END), 0), 2) AS net_yoy_growth_pct,
  COUNT(DISTINCT CASE WHEN in_ttm   THEN order_id END) AS order_count_ttm,
  COUNT(DISTINCT CASE WHEN in_prior THEN order_id END) AS order_count_prior_ttm,
  ROUND(100 * (COUNT(DISTINCT CASE WHEN in_ttm   THEN order_id END)
             - COUNT(DISTINCT CASE WHEN in_prior THEN order_id END))
        / NULLIF(COUNT(DISTINCT CASE WHEN in_prior THEN order_id END), 0), 2) AS order_yoy_growth_pct
FROM lines
GROUP BY country, city

In [0]:
%sql
-- Verification: shape, totals, integrity. Expected: 11 rows = 11 distinct
-- geographies across 4 countries, SUM(buying_companies) = 390, SUM(order_count)
-- = 10,817, SUM(net_revenue) = 28,333,498.48 and SUM(gross_revenue) =
-- 30,723,269.53 (exact KPI 1-6 reconciliation), share sums to 100.00, 0 null or
-- empty geographies, 0 growth-formula inconsistencies (growth % recomputed
-- from the stored window columns must match the stored pct).
SELECT
  COUNT(*)                                                                  AS rows_geographies,
  COUNT(DISTINCT country || ' | ' || city)                                  AS distinct_geographies,
  COUNT(DISTINCT country)                                                   AS countries,
  SUM(buying_companies)                                                     AS total_buying_companies,
  SUM(order_count)                                                          AS total_orders,
  ROUND(SUM(net_revenue), 2)                                                AS sum_net,
  ROUND(SUM(gross_revenue), 2)                                              AS sum_gross,
  ROUND(SUM(net_revenue_share_pct), 2)                                      AS sum_share_pct,
  SUM(CASE WHEN country IS NULL OR city IS NULL OR TRIM(city) = ''
           THEN 1 ELSE 0 END)                                               AS bad_geo_rows,
  SUM(CASE WHEN net_revenue IS NULL OR gross_revenue IS NULL
           THEN 1 ELSE 0 END)                                                AS null_revenue_rows,
  SUM(CASE WHEN net_yoy_growth_pct IS NOT NULL
            AND ABS(net_yoy_growth_pct - ROUND(100 * (net_ttm - net_prior_ttm)
                 / NULLIF(net_prior_ttm, 0), 2)) > 0.005
           THEN 1 ELSE 0 END)                                               AS growth_mismatches,
  SUM(CASE WHEN order_yoy_growth_pct IS NOT NULL
            AND ABS(order_yoy_growth_pct - ROUND(100 * (order_count_ttm - order_count_prior_ttm)
                 / NULLIF(order_count_prior_ttm, 0), 2)) > 0.005
           THEN 1 ELSE 0 END)                                               AS order_growth_mismatches
FROM b2b_ecommerce.gold.kpi_7_geographic_sales

In [0]:
%sql
-- Verification: independently recompute every geography through a different
-- path - per-company_sk subquery first, then rolled up to (country, city); no
-- shared CTEs with the view - and cross-check the TTM/prior windows at whole-
-- fact level. Expected: geographies_checked = 11, every match count = 11,
-- view window totals = fact-level window totals exactly, 0 join mismatches.
WITH fact AS (
  SELECT company_sk, order_id, status, line_total, order_date
  FROM b2b_ecommerce.gold.fact_order_items
  WHERE order_date_valid = true
),
bounds AS (
  SELECT DATE_TRUNC('month', MAX(order_date)) AS trunc_month FROM fact
),
per_company AS (
  SELECT
    f.company_sk,
    COUNT(DISTINCT f.order_id) AS order_count,
    SUM(f.line_total) AS gross_amt,
    SUM(CASE WHEN f.status <> 'cancelled' THEN f.line_total ELSE 0 END) AS net_amt,
    SUM(CASE WHEN f.status <> 'cancelled'
              AND f.order_date >= ADD_MONTHS(b.trunc_month, -12)
              AND f.order_date <  b.trunc_month
             THEN f.line_total ELSE 0 END) AS net_ttm,
    SUM(CASE WHEN f.status <> 'cancelled'
              AND f.order_date >= ADD_MONTHS(b.trunc_month, -24)
              AND f.order_date <  ADD_MONTHS(b.trunc_month, -12)
             THEN f.line_total ELSE 0 END) AS net_prior,
    COUNT(DISTINCT CASE WHEN f.order_date >= ADD_MONTHS(b.trunc_month, -12)
                          AND f.order_date <  b.trunc_month
                         THEN f.order_id END) AS orders_ttm,
    COUNT(DISTINCT CASE WHEN f.order_date >= ADD_MONTHS(b.trunc_month, -24)
                          AND f.order_date <  ADD_MONTHS(b.trunc_month, -12)
                         THEN f.order_id END) AS orders_prior
  FROM fact f
  CROSS JOIN bounds b
  GROUP BY f.company_sk
),
recompute AS (
  SELECT
    d.country,
    d.city,
    COUNT(*)                 AS buying_companies,
    SUM(p.order_count)       AS order_count,
    ROUND(SUM(p.gross_amt), 2) AS gross_revenue,
    ROUND(SUM(p.net_amt), 2)   AS net_revenue,
    ROUND(SUM(p.net_ttm), 2)   AS net_ttm,
    ROUND(SUM(p.net_prior), 2) AS net_prior_ttm,
    SUM(p.orders_ttm)       AS order_count_ttm,
    SUM(p.orders_prior)     AS order_count_prior_ttm
  FROM per_company p
  JOIN b2b_ecommerce.gold.dim_companies d
    ON d.company_sk = p.company_sk
   AND d.__END_AT IS NULL
  GROUP BY d.country, d.city
),
fact_totals AS (
  SELECT
    COUNT(DISTINCT CASE WHEN f.order_date >= ADD_MONTHS(b.trunc_month, -12)
                          AND f.order_date <  b.trunc_month
                         THEN f.order_id END)          AS fact_orders_ttm,
    COUNT(DISTINCT CASE WHEN f.order_date >= ADD_MONTHS(b.trunc_month, -24)
                          AND f.order_date <  ADD_MONTHS(b.trunc_month, -12)
                         THEN f.order_id END)          AS fact_orders_prior,
    ROUND(SUM(CASE WHEN f.status <> 'cancelled'
                    AND f.order_date >= ADD_MONTHS(b.trunc_month, -12)
                    AND f.order_date <  b.trunc_month
                   THEN f.line_total ELSE 0 END), 2)   AS fact_net_ttm,
    ROUND(SUM(CASE WHEN f.status <> 'cancelled'
                    AND f.order_date >= ADD_MONTHS(b.trunc_month, -24)
                    AND f.order_date <  ADD_MONTHS(b.trunc_month, -12)
                   THEN f.line_total ELSE 0 END), 2)   AS fact_net_prior
  FROM fact f
  CROSS JOIN bounds b
)
SELECT
  COUNT(*)                          AS geographies_checked,
  SUM(CASE WHEN v.buying_companies = r.buying_companies THEN 1 ELSE 0 END)       AS match_companies,
  SUM(CASE WHEN v.order_count = r.order_count THEN 1 ELSE 0 END)                 AS match_orders,
  SUM(CASE WHEN v.gross_revenue = r.gross_revenue THEN 1 ELSE 0 END)             AS match_gross,
  SUM(CASE WHEN v.net_revenue = r.net_revenue THEN 1 ELSE 0 END)                 AS match_net,
  SUM(CASE WHEN v.net_ttm = r.net_ttm THEN 1 ELSE 0 END)                         AS match_net_ttm,
  SUM(CASE WHEN v.net_prior_ttm = r.net_prior_ttm THEN 1 ELSE 0 END)             AS match_net_prior_ttm,
  SUM(CASE WHEN v.order_count_ttm = r.order_count_ttm THEN 1 ELSE 0 END)         AS match_orders_ttm,
  SUM(CASE WHEN v.order_count_prior_ttm = r.order_count_prior_ttm THEN 1 ELSE 0 END) AS match_orders_prior_ttm,
  SUM(CASE WHEN v.country IS NULL OR r.country IS NULL
           THEN 1 ELSE 0 END)                                                     AS join_mismatch_rows,
  SUM(v.order_count_ttm)                                                          AS view_orders_ttm,
  ft.fact_orders_ttm,
  SUM(v.order_count_prior_ttm)                                                   AS view_orders_prior_ttm,
  ft.fact_orders_prior,
  ROUND(SUM(v.net_ttm), 2)                                                       AS view_net_ttm,
  ft.fact_net_ttm,
  ROUND(SUM(v.net_prior_ttm), 2)                                                 AS view_net_prior_ttm,
  ft.fact_net_prior
FROM b2b_ecommerce.gold.kpi_7_geographic_sales v
FULL OUTER JOIN recompute r
  ON v.country = r.country AND v.city = r.city
CROSS JOIN fact_totals ft
GROUP BY ft.fact_orders_ttm, ft.fact_orders_prior, ft.fact_net_ttm, ft.fact_net_prior
-- (single-row GROUP BY on the fact-level constants so the aggregate columns and
-- the cross-joined totals can share one SELECT)

In [0]:
%sql
-- Country rollup (GROUP BY over the view): companies, orders, revenue and
-- trailing-vs-prior-12-full-months growth at country grain.
SELECT
  country,
  COUNT(*)                        AS cities,
  SUM(buying_companies)            AS buying_companies,
  SUM(order_count)                 AS orders,
  SUM(gross_revenue)               AS gross_revenue,
  SUM(net_revenue)                 AS net_revenue,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS net_share_pct,
  SUM(net_ttm)                     AS net_ttm,
  SUM(net_prior_ttm)               AS net_prior_ttm,
  ROUND(100 * (SUM(net_ttm) - SUM(net_prior_ttm))
        / NULLIF(SUM(net_prior_ttm), 0), 2)                          AS net_yoy_growth_pct
FROM b2b_ecommerce.gold.kpi_7_geographic_sales
GROUP BY country
ORDER BY net_revenue DESC

In [0]:
%sql
-- Growth regions: all geographies ranked by net YoY growth (trailing 12 full
-- months vs prior 12). NULL prior = region too new for a comparable base.
SELECT country, city, buying_companies, order_count,
       net_revenue, net_revenue_share_pct,
       net_ttm, net_prior_ttm, net_yoy_growth_pct, order_yoy_growth_pct
FROM b2b_ecommerce.gold.kpi_7_geographic_sales
ORDER BY net_yoy_growth_pct DESC, net_revenue DESC

In [0]:
%sql
-- Top 10 geographies by net revenue (headline view).
SELECT country, city, buying_companies, order_count, gross_revenue, net_revenue,
       net_revenue_share_pct, net_yoy_growth_pct
FROM b2b_ecommerce.gold.kpi_7_geographic_sales
ORDER BY net_revenue DESC
LIMIT 10